<a href="https://colab.research.google.com/github/frayyan-rgb/amazon-employee-feedback-analysis/blob/main/09_segment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

themes = pd.read_csv("combined_with_themes.csv")
seg = pd.read_csv("Project5_Labeled_Segments.csv")[["review_id", "Segment"]]

df = themes[themes["platform"] == "Glassdoor"].copy()   # drops the 9 YouTube rows
df["review_id"] = df["review_id"].astype(int)
df = df.merge(seg, on="review_id", how="left")

print(df.shape, "| missing Segment:", df["Segment"].isna().sum())  # expect (139, 41) | 0

(139, 41) | missing Segment: 0


In [2]:
THEMES = ["Shift Strain (Night vs Day)", "Burnout & Fatigue", "New-Hire Training & Support"]

df["Theme"] = df["Theme"].fillna("")
for t in THEMES:
    df[t] = df["Theme"].str.contains(t, regex=False).astype(int)

df["Any Theme"] = (df[THEMES].sum(axis=1) > 0).astype(int)

In [3]:
order = df["Segment"].value_counts().index          # biggest segment first

counts = df.groupby("Segment")[THEMES].sum().loc[order]
counts["Reviews with any theme"] = df.groupby("Segment")["Any Theme"].sum().loc[order]
counts["Total reviews"] = df["Segment"].value_counts().loc[order]
counts.loc["ALL SEGMENTS"] = counts.sum()

counts

,Shift Strain (Night vs Day),Burnout & Fatigue,New-Hire Training & Support,Reviews with any theme,Total reviews
Segment,,,,,
Full-Time Associates,16,12,4,26,77
Part-Time & Seasonal Associates,3,1,2,6,25
Associates (Status Not Reported),5,2,2,6,15
Managers & Corporate Roles,2,1,1,3,12
"Contract, Intern & Freelance",2,0,0,2,10
ALL SEGMENTS,28,16,9,43,139


In [4]:
share = (counts[THEMES].div(counts["Total reviews"], axis=0) * 100).round(0).astype(int)
share.columns = [c + " (%)" for c in THEMES]

share

,Shift Strain (Night vs Day) (%),Burnout & Fatigue (%),New-Hire Training & Support (%)
Segment,,,
Full-Time Associates,21,16,5
Part-Time & Seasonal Associates,12,4,8
Associates (Status Not Reported),33,13,13
Managers & Corporate Roles,17,8,8
"Contract, Intern & Freelance",20,0,0
ALL SEGMENTS,20,12,6


In [5]:
counts.to_csv("Theme_Frequency_by_Segment.csv")
share.to_csv("Theme_Share_by_Segment.csv")